In [8]:
!uv sync --group dev

/Users/rayan/learning/git/ai-playground/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

/Users/rayan/learning/git/ai-playground/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1123 13:59:02.282000 66776 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Read training dataset

In [2]:
df = pd.read_csv("./data/intent_training_samples.csv")
df.head(5)

,intent,prompt
0,culinary_assistance,Can you explain how to replace buttermilk in a...
1,out_of_scope,What should I know about understanding how mor...
2,recipe_lookup,Please recommend something for a Mexican meal ...
3,recipe_lookup,What could I cook for a mild curry that uses c...
4,recipe_lookup,Please recommend something for an easy breakfa...


## Create label to number mappings

In [4]:
label2id = { "recipe_lookup": 0, "culinary_assistance": 1, "out_of_scope": 2 }
id2label = { 0: "recipe_lookup", 1: "culinary_assistance", 2: "out_of_scope" }

## Add label to dataframe

In [5]:
df["label"] = df["intent"].map(label2id)
df.head

<bound method NDFrame.head of                   intent                                             prompt  \
0    culinary_assistance  Can you explain how to replace buttermilk in a...   
1           out_of_scope  What should I know about understanding how mor...   
2          recipe_lookup  Please recommend something for a Mexican meal ...   
3          recipe_lookup  What could I cook for a mild curry that uses c...   
4          recipe_lookup  Please recommend something for an easy breakfa...   
..                   ...                                                ...   
295  culinary_assistance  Please help me scale a soup recipe from four s...   
296  culinary_assistance  How can I make bread dough rise properly in a ...   
297  culinary_assistance  How can I adjust the baking time for cupcakes ...   
298        recipe_lookup  Please recommend something for a gluten-free d...   
299        recipe_lookup  Please recommend something for a refreshing su...   

     label  
0       

## Split traning and test data

In [6]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 240
Test size: 60


## Convert Pandas df to Hugging Face ds

In [8]:
train_dataset = Dataset.from_pandas(
    train_df[["prompt", "label"]],
    preserve_index=False
    )
test_dataset = Dataset.from_pandas(
    test_df[["prompt", "label"]],
    preserve_index=False
    )

## Load DistilBERT tokenizer

In [9]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [10]:
tokenizer

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

## Tokenize complete dataset (training & test)

In [11]:
def tokenize_function(batch):
    return tokenizer(batch["prompt"], padding="max_length", truncation=True)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 60/60 [00:00<00:00, 7241.96 examples/s]


## Load DistilBERT for classificatin

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,  # len(label2id),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 10361.42it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Configure Training parameters

In [20]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,

    per_device_train_batch_size=10,
    per_device_eval_batch_size=10,
    
    num_train_epochs=3,
    weight_decay=0.01,
    
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    report_to="none"
)


## Create Trainer

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_test_dataset,
    processing_class=tokenizer,
    compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))}
)

## Fine-tune Dataset

In [22]:
trainer.train()

/Users/rayan/learning/git/ai-playground/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.017973,0.009469,1.000000
2,0.009047,0.005702,1.000000
3,0.006832,0.004951,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.80it/s]
/Users/rayan/learning/git/ai-playground/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 12.54it/s]
/Users/rayan/learning/git/ai-playground/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.45it/s]


TrainOutput(global_step=72, training_loss=0.01321503183700972, metrics={'train_runtime': 34.267, 'train_samples_per_second': 21.011, 'train_steps_per_second': 2.101, 'total_flos': 95378227937280.0, 'train_loss': 0.01321503183700972, 'epoch': 3.0})

## Measure training Accuracy

In [19]:
predictions = trainer.predict(tokenized_test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(
    y_true,
    y_pred,
    target_names=[id2label[i] for i in range(len(id2label))],
    zero_division=0,
))

/Users/rayan/learning/git/ai-playground/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Accuracy: 1.0
                     precision    recall  f1-score   support

      recipe_lookup       1.00      1.00      1.00        20
culinary_assistance       1.00      1.00      1.00        20
       out_of_scope       1.00      1.00      1.00        20

           accuracy                           1.00        60
          macro avg       1.00      1.00      1.00        60
       weighted avg       1.00      1.00      1.00        60



## Create inference Function to test trained model

In [23]:
def classify_intent(user_input):
    
    inputs = tokenizer(user_input, return_tensors="pt", padding=True, truncation=True)
    
    # Move input tensors to same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()

    with torch.no_grad():
        
        outputs = model(**inputs)
        
        logits = outputs.logits
        # predicted_class_id = torch.argmax(logits, dim=1).item()
        # predicted_label = id2label[predicted_class_id]

        probabilities = torch.softmax(logits, dim=-1)[0]
    
    predicted_id = torch.argmax(probabilities).item()
    predicted_intent = id2label[predicted_id]
    confidence = probabilities[predicted_id].item()

    return predicted_intent, confidence


## Test the trained model

In [24]:
sample_prompts = [
    "Suggest a vegetarian pasta recipe for dinner",
    "How do I stop scrambled eggs from becoming rubbery?",
    "What is the difference between REST and GraphQL?",
    "How can I substitute cream in a soup?",
    "Help me plan a healthy weeknight meal",
]

for prompt in sample_prompts:
    predicted_intent, confidence = classify_intent(prompt)
    print(f"Prompt: {prompt}")
    print(f"Intent: {predicted_intent} ({confidence:.2%})\n")

Prompt: Suggest a vegetarian pasta recipe for dinner
Intent: recipe_lookup (99.04%)

Prompt: How do I stop scrambled eggs from becoming rubbery?
Intent: culinary_assistance (98.75%)

Prompt: What is the difference between REST and GraphQL?
Intent: out_of_scope (72.81%)

Prompt: How can I substitute cream in a soup?
Intent: culinary_assistance (99.05%)

Prompt: Help me plan a healthy weeknight meal
Intent: out_of_scope (91.12%)



## Save the trained MODEL

In [25]:
trainer.model.save_pretrained("./data/intent-model")
tokenizer.save_pretrained("./data/intent-model")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.13it/s]


('./data/intent-model/tokenizer_config.json',
 './data/intent-model/tokenizer.json')

## Save to Huggingface Repo

In [30]:
!uv add huggingface_hub --upgrade

Resolved 163 packages in 453ms                                       
⠙ Preparing packages... (0/47)                                                  
⠙ Preparing packages... (0/47)------------------     0 B/95.60 KiB           
⠙ Preparing packages... (0/47)------------------     0 B/95.60 KiB           
pycparser            ------------------------------     0 B/61.13 KiB
⠙ Preparing packages... (0/47)------------------     0 B/95.60 KiB           
pycparser            ------------------------------     0 B/61.13 KiB
⠙ Preparing packages... (0/47)------------------     0 B/95.60 KiB           
pycparser            ------------------------------     0 B/61.13 KiB
⠙ Preparing packages... (0/47)------------------     0 B/95.60 KiB           
python-multipart     ------------------------------     0 B/31.82 KiB
pycparser            ------------------------------     0 B/61.13 KiB
⠙ Preparing packages... (0/47)------------------     0 B/95.60 KiB           
python-multipart     ----------

In [32]:
!uv run hf auth login

⠙ Preparing packages... (0/6)                                                   
⠙ Preparing packages... (0/6)-------------------     0 B/90.68 KiB           
⠙ Preparing packages... (0/6)-------------------     0 B/90.68 KiB           
python-dateutil      ------------------------------     0 B/90.68 KiB
⠙ Preparing packages... (0/6)-------------------     0 B/340.05 KiB          
python-dateutil      ------------------------------ 14.76 KiB/90.68 KiB
⠙ Preparing packages... (0/6)-------------------     0 B/340.05 KiB          
python-dateutil      ------------------------------ 14.76 KiB/90.68 KiB
⠙ Preparing packages... (0/6)------------------- 14.78 KiB/340.05 KiB        
python-dateutil      ------------------------------ 14.76 KiB/90.68 KiB
⠙ Preparing packages... (0/6)------------------- 14.78 KiB/340.05 KiB        
python-dateutil      ------------------------------ 14.76 KiB/90.68 KiB
⠙ Preparing packages... (0/6)------------------- 14.78 KiB/340.05 KiB        
python-dateutil

In [34]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

repo_id = "get2rayan/smartsavor-intent-distilbert"

# load from local training output directory
model = AutoModelForSequenceClassification.from_pretrained("./data/intent-model")
tokenizer = AutoTokenizer.from_pretrained("./data/intent-model")

# Push to Hugging Face Hub
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 12732.64it/s]


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 25.09it/s]
Processing Files (1 / 1): 100%|██████████|  268MB /  268MB,   ???B/s  






Processing Files (1 / 1): 100%|██████████|  268MB /  268MB, 25.3MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Validating: ██████████| 100%            
No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/get2rayan/smartsavor-intent-distilbert/commit/86622e1035396bcdd9c44c38c81544b37a734f34', commit_message='Upload tokenizer', commit_description='', oid='86622e1035396bcdd9c44c38c81544b37a734f34', pr_url=None, repo_url=RepoUrl('https://huggingface.co/get2rayan/smartsavor-intent-distilbert', endpoint='https://huggingface.co', repo_type='model', repo_id='get2rayan/smartsavor-intent-distilbert'), pr_revision=None, pr_num=None)